# MÓDULO 2
## Errores y excepciones en Python (gestión y creación)

**Objetivos:**
- Diferenciar **errores** y **excepciones**.
- Aprender a **leer un traceback** (pila de llamadas).
- Controlar fallos con `try` / `except` / `else` / `finally`.
- Lanzar excepciones con `raise` y **encadenarlas** correctamente.
- Crear **excepciones propias** (casos de negocio).
- Ver un ejemplo realista: **creación de pizzas válidas**.
- Conocer novedades relevantes en Python moderno (3.11+).

> Documentación oficial (muy recomendable):
> - Tutorial: *Errors and Exceptions* (Python)  
>   https://docs.python.org/3/tutorial/errors.html
> - Referencia: *Built-in Exceptions*  
>   https://docs.python.org/3/library/exceptions.html


## Índice
- Errores vs excepciones
- Leer un traceback
- Excepciones comunes (las que verás a diario)
- `try` / `except`: capturar y continuar
- Varios `except`, `else` y `finally`
- Lanzar excepciones con `raise`
- Buenas prácticas
- Excepciones propias (clases)
- Caso práctico: “solo se pueden crear ciertas pizzas”
- Novedades Python 3.11+: `ExceptionGroup`, `except*`, `add_note`
- Ejercicios


## Errores vs excepciones

En Python solemos hablar de “errores” de forma genérica, pero conviene distinguir:

### 1) Errores de sintaxis (SyntaxError)
Son fallos **en la forma del código**. El programa **ni siquiera empieza**:
- Falta de `:` al final de un `if`
- Paréntesis sin cerrar
- Indentación incorrecta

In [ ]:
print("inicio")
print("hola"
print("final")

### 2) Excepciones (exceptions)
Son fallos que ocurren **mientras el programa ya está ejecutándose** (en tiempo de ejecución):
- Dividir entre cero
- Convertir `"hola"` a `int`
- Acceder a una clave que no existe en un diccionario
- Abrir un archivo que no existe

Las **excepciones** son objetos (instancias de clases que heredan de `Exception`) que describen qué ha salido mal.


In [ ]:
# Ejemplo típico: una excepción no controlada.
# Ejecuta esta celda para ver el traceback.

resultado = 1 / 0
print(resultado)

## Leer un traceback (pila de llamadas)

Cuando ocurre una excepción *no controlada*, Python imprime un **traceback**:
- La lista de llamadas (qué funciones se estaban ejecutando)
- La línea exacta donde se produjo el problema
- El tipo de excepción y un mensaje

Esto es oro para depurar: te dice *dónde* y *por qué* falló.

Si la celda anterior de la división por 0 “rompe”, es normal: hemos provocado una excepción a propósito.

La idea clave de este tema es:
- Si NO controlas la excepción: el programa se detiene (o, en un notebook, esa celda falla).
- Si SÍ la controlas: capturas el problema y decides qué hacer.

## Excepciones comunes (muy importantes)

Estas son de las más frecuentes en proyectos reales:

- `ValueError`: el valor no es válido (p.ej., `int("hola")`).
- `TypeError`: el tipo no encaja (p.ej., `"3" + 2`).
- `KeyError`: clave inexistente en un diccionario.
- `IndexError`: índice fuera de rango en una lista.
- `AttributeError`: atributo/método que no existe.
- `FileNotFoundError`: archivo inexistente.
- `ZeroDivisionError`: división entre cero.

Vamos a ver ejemplos breves.


In [ ]:
# ValueError
try:
    int("hola")
except ValueError as e:
    print("ValueError capturada:", e)


In [ ]:
# TypeError
try:
    "3" + 2
except TypeError as e:
    print("TypeError capturada:", e)


In [ ]:
# KeyError
d = {"a": 1}
try:
    d["b"]
except KeyError as e:
    print("KeyError capturada:", e)


In [ ]:
# IndexError
xs = [10, 20]
try:
    xs[10]
except IndexError as e:
    print("IndexError capturada:", e)


In [ ]:
# AttributeError
try:
    (123).upper()
except AttributeError as e:
    print("AttributeError capturada:", e)


## `try` / `except`: capturar y continuar

La estructura básica es:

```python
try:
    # código que podría fallar
except TipoDeExcepcion:
    # qué hacemos si falla
```

Esto es correcto cuando:
- El fallo es **esperable** (entrada del usuario, datos externos, red, archivos…)
- Queremos **seguir** y tomar una decisión (mensaje, reintento, fallback…)


In [ ]:
# Sin try/except: si falla, se detiene (la celda falla).
# Con try/except: capturamos el fallo y continuamos.

print("Antes del try")

try:
    x = int("no es un número")
    print("Esto no se ejecuta si falla la conversión")
except ValueError:
    print("No se pudo convertir a entero. Continuamos...")

print("Después del try (se ejecuta)")


## Capturar el objeto excepción (`as e`)

Capturar `as e` te permite:
- Mostrar un mensaje más informativo
- Guardar detalles en logs
- Tomar decisiones según el contenido


In [ ]:
try:
    int("12.3")
except ValueError as e:
    print("Tipo:", type(e))
    print("Mensaje:", str(e))


## Varios `except` (según el tipo de fallo)

Puedes distinguir distintos problemas y tratarlos de forma diferente.


In [ ]:
def dividir(a, b):
    return a / b

pares = [("10", 2), (10, 0), ("hola", 3)]

for a, b in pares:
    try:
        print(dividir(int(a), b))
    except ValueError:
        print("Entrada inválida: a debe ser un entero.")
    except ZeroDivisionError:
        print("No se puede dividir entre cero.")

## Capturar varias excepciones a la vez

Útil cuando el tratamiento es el mismo. Intetamos convertir un valor a int y devolvemos None si falla



In [1]:
def parsear_entero_o_none(texto):
    try:
        return int(texto)
    except (TypeError, ValueError):
        return None

print(parsear_entero_o_none("123"))
print(parsear_entero_o_none("12.3"))
print(parsear_entero_o_none(None))


123
None
None


## `else`: cuando NO hubo excepción

`else` se ejecuta solo si el `try` terminó bien. Es muy útil para:
- Dejar el `try` lo más pequeño posible.
- Separar la parte “peligrosa” de la parte “normal”.


In [ ]:
texto = "42"

try:
    n = int(texto)
except ValueError:
    print("No es un entero.")
else:
    print("Conversión correcta. n =", n)


## `finally`: se ejecute o no se ejecute, esto siempre ocurre

`finally` se ejecuta SIEMPRE (haya excepción o no). Se usa para:
- Cerrar recursos (archivos, conexiones…)
- Liberar memoria / limpiar estados
- Garantizar consistencia

Nota: en muchos casos `with` es mejor que `finally` para recursos.


In [ ]:
def dividir_con_mensaje(a, b):
    print("Inicio de la operación")
    try:
        return a / b
    except ZeroDivisionError:
        return "No se puede dividir entre cero"
    finally:
        print("Fin de la operación")

print(dividir_con_mensaje(10, 2))
print(dividir_con_mensaje(10, 0))

## Lanzar excepciones con `raise`

A veces el fallo no viene de Python, sino de **tus reglas**.

Ejemplo: una función espera un entero positivo. Si no lo es, lo correcto es **lanzar una excepción**:
- El error se detecta justo donde ocurre el problema
- Se obliga a quien llama a decidir qué hacer (capturar, propagar, etc.)


In [ ]:
def raiz_cuadrada(n):
    if not isinstance(n, (int, float)):
        raise TypeError("n debe ser int o float")
    if n < 0:
        raise ValueError("n debe ser >= 0")
    return n ** 0.5

print(raiz_cuadrada(9))


Prueba a llamar `raiz_cuadrada(-1)` o `raiz_cuadrada("9")` y verás cómo `raise` comunica claramente el problema.

## Relanzar y encadenar excepciones

### Relanzar (re-raise)
Si capturas una excepción pero decides que no puedes resolverla, puedes relanzarla:

```python
except ValueError:
    ...
    raise
```

### Encadenar (`raise ... from e`)
Cuando transformas una excepción en otra, conviene conservar la causa original:
- Esto crea un traceback más informativo.


In [ ]:
def convertir_y_dividir(texto, divisor):
    try:
        n = int(texto)
    except ValueError as e:
        raise ValueError(f"No se pudo convertir '{texto}' a entero") from e
    return n / divisor

try:
    convertir_y_dividir("12.3", 2)
except ValueError as e:
    print("Error:", e)
    # La causa original sigue existiendo:
    print("Causa original:", repr(e.__cause__))


## Buenas prácticas (importantes)

- Captura **excepciones específicas** (mejor que `except Exception:`).
- Evita `except:` a secas (captura demasiadas cosas, incluso `KeyboardInterrupt` en algunos contextos).
- Mantén el bloque `try` **pequeño**: solo lo que puede fallar.
- No uses excepciones para controlar lo “normal” del flujo (no abuses).
- No hagas `except ...: pass` sin una razón muy clara.
- Para validaciones de entrada/negocio, `raise ValueError` o excepciones propias suele ser lo adecuado.


## Crear excepciones propias (muy útil)

Cuando tu programa tiene reglas de negocio, es común crear excepciones con nombres claros.

Regla:
- Hereda de `Exception` (normalmente).
- Usa nombres que describan el problema.


In [1]:
class PizzaError(Exception):
    """Excepción base para errores relacionados con pizzas."""

class TipoPizzaInvalido(PizzaError):
    """Se pidió un tipo de pizza que no existe en el sistema."""

class TamanoPizzaInvalido(PizzaError):
    """Tamaño no permitido para ese tipo de pizza."""

## Caso práctico: crear pizzas válidas

Vamos a construir una función `crear_pizza(...)` con reglas claras:

- Solo hay ciertos tipos de pizza.
- Hay tamaños permitidos.
- Si algo no cumple, **lanzamos una excepción**.

¿Por qué es correcto usar excepciones aquí?
- Porque “pedir una pizza que no existe” no es un caso normal, es un **error de entrada**.
- Queremos informar del fallo *sin que el programa se detenga* (si lo gestionamos).


In [2]:
MENU = {
    "margarita": {"mediana", "familiar"},
    "barbacoa": {"pequena", "mediana", "familiar"},
    "cuatro quesos": {"mediana"},
}

def crear_pizza(tipo, tamano):
    tipo = tipo.strip().casefold()
    tamano = tamano.strip().casefold()

    if tipo not in MENU:
        raise TipoPizzaInvalido(
            f"Tipo de pizza inválido: {tipo!r}. Tipos disponibles: {sorted(MENU)}"
        )

    tamanos_permitidos = MENU[tipo]
    if tamano not in tamanos_permitidos:
        raise TamanoPizzaInvalido(
            f"Tamaño inválido {tamano!r} para {tipo!r}. Tamaños permitidos: {sorted(tamanos_permitidos)}"
        )

    return f" > Pizza creada: {tipo} ({tamano})"


### Probar un caso correcto

In [3]:
print(crear_pizza("Margarita", "Mediana"))

 > Pizza creada: margarita (mediana)


### Probar casos incorrectos

La idea es que **no se detenga tu programa completo**: capturas la excepción y decides qué hacer.


In [4]:
# Tipo inválido: capturamos y seguimos
try:
    print(crear_pizza("hawaiiana", "mediana"))
except TipoPizzaInvalido as e:
    print("Tipo no válido:", e)

print("El programa sigue ejecutándose...")

Tipo no válido: Tipo de pizza inválido: 'hawaiiana'. Tipos disponibles: ['barbacoa', 'cuatro quesos', 'margarita']
El programa sigue ejecutándose...


In [5]:
# Tamaño inválido: capturamos y seguimos
try:
    print(crear_pizza("cuatro quesos", "familiar"))
except TamanoPizzaInvalido as e:
    print("Tamaño no válido:", e)

print("El programa sigue ejecutándose...")

Tamaño no válido: Tamaño inválido 'familiar' para 'cuatro quesos'. Tamaños permitidos: ['mediana']
El programa sigue ejecutándose...


## ¿Qué hago cuando capturo una excepción?

Depende del contexto. Algunas opciones típicas:

1) Mostrar un mensaje y terminar esa operación (pero el programa sigue).
2) Reintentar con otros parámetros (si tiene sentido).
3) Aplicar un “plan B” (fallback).
4) Registrar el error (logging) y continuar.

Ejemplo sencillo de “reintento” sin interacción:


In [ ]:
intentos = [
    {"tipo": "carbonara", "tamano": "mediana"},      # fallará (tipo inválido)
    {"tipo": "cuatro quesos", "tamano": "familiar"}, # fallará (tamaño inválido)
    {"tipo": "cuatro quesos", "tamano": "mediana"},  # correcto
]

for i, params in enumerate(intentos, start=1):
    try:
        p = crear_pizza(**params) # **params desempaqueta el contenido del diccionario, equivale a: p = crear_pizza(tipo="carbonara", tamano="mediana")
    except PizzaError as e:
        print(f"Intento {i} falló:", e)
    else:
        print(f"Intento {i} OK ->", p)
        break

Intento 0 falló: Tipo de pizza inválido: 'carbonara'. Tipos disponibles: ['barbacoa', 'cuatro quesos', 'margarita']
Intento 1 falló: Tamaño inválido 'familiar' para 'cuatro quesos'. Tamaños permitidos: ['mediana']
Intento 2 OK ->  > Pizza creada: cuatro quesos (mediana)


Observa que:
- Si falla, **capturamos** y seguimos al siguiente intento.
- Si va bien, entramos en `else` y salimos.

Este patrón es muy común cuando dependes de recursos externos o validaciones. Pero también podemos intentar dar solución al problema:


In [8]:
tipo = input("Tipo: ") # margarita, barbacoa, cuatro quesos
tamano = input("Tamaño: ") # pequena, mediana, familiar

while True:
    try:
        print(crear_pizza(tipo, tamano))
        break
    except TipoPizzaInvalido as e:
        print(e)
        tipo = input("Tipo (corrige): ")
    except TamanoPizzaInvalido as e:
        print(e)
        tamano = input("Tamaño (corrige): ")


Tipo de pizza inválido: 'trufa'. Tipos disponibles: ['barbacoa', 'cuatro quesos', 'margarita']
 > Pizza creada: barbacoa (mediana)


### Otro ejemplo completo

In [ ]:
class EdadInvalida(Exception):
    """Edad no válida (formato o rango)."""

def pedir_edad():
    while True:
        try:
            entrada = input("Introduce tu edad (0-120): ").strip()

            # 1) Convertir a int (puede lanzar ValueError)
            edad = int(entrada)

            # 2) Validar rango (lanzamos nuestra excepción)
            if not (0 <= edad <= 120):
                raise EdadInvalida("La edad debe estar entre 0 y 120.")

            return edad

        except ValueError:
            print(" > Debes escribir un número entero (ej: 18).")
        except EdadInvalida as e:
            print(" >", e)


edad = pedir_edad()
print(" > Edad válida:", edad)


 > La edad debe estar entre 0 y 120.
 > Edad válida: 20


## Nota: `assert` vs excepciones

`assert` se usa para verificar **invariantes internas** (cosas que, si fallan, indican un bug del programador).
No es lo ideal para validar entrada del usuario.

- Para entrada de usuario / negocio → `raise ValueError`, `TypeError` o excepción propia.
- Para “esto no debería pasar nunca” → `assert`.


In [ ]:
def media(xs):
    assert len(xs) > 0, "La lista no puede estar vacía (invariante interna)"
    return sum(xs) / len(xs)

print(media([1, 2, 3]))

## Novedades en Python 3.11+ (opcional, pero actual)

### 1) `ExceptionGroup` y `except*`
Sirve para manejar múltiples errores que ocurren “a la vez” (muy útil en concurrencia).

Importante: `except*` es sintaxis nueva. Solo si tu Python es 3.11+.


In [ ]:
def demo_exception_group():
    eg = ExceptionGroup(
        "Varios problemas",
        [ValueError("valor mal"), TypeError("tipo mal"), ValueError("otro valor mal")]
    )
    raise eg

try:
    demo_exception_group()
except* ValueError as eg_val:
    print("Capturé ValueError(s):", [str(e) for e in eg_val.exceptions])
except* TypeError as eg_type:
    print("Capturé TypeError(s):", [str(e) for e in eg_type.exceptions])

### 2) `BaseException.add_note()` (Python 3.11+)

Permite añadir notas a una excepción para enriquecer el traceback (útil en debugging).
Este ejemplo también se ejecuta solo en Python 3.11+.


In [ ]:
import sys

if sys.version_info >= (3, 11):
    try:
        int("12.3")
    except ValueError as e:
        e.add_note("Consejo: si esperas decimales, usa float(...) en lugar de int(...).")
        raise
else:
    print("add_note requiere Python 3.11+ (tu versión es", sys.version, ")")
